In [2]:
!pip install pyspark
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("UK Online Retail Analytics")
    .master("local[*]")
    .getOrCreate()
)

In [5]:
spark

In [6]:
df = spark.read.parquet("/content/drive/MyDrive/Pyspark_Project/output/cleaned_data")

In [7]:
df.show(10)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+----+-----+---------+---+-------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|TotalAmount|Year|Month|MonthName|Day|Weekday|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+----+-----+---------+---+-------+
|   536373|    21071|VINTAGE BILLBOARD...|       6|2010-12-01 09:02:00|     1.06|     17850|United Kingdom|       6.36|2010|   12|      Dec|  1|      2|
|   536373|   84029G|KNITTED UNION FLA...|       6|2010-12-01 09:02:00|     3.39|     17850|United Kingdom|      20.34|2010|   12|      Dec|  1|      2|
|   536412|    20727|LUNCH BAG  BLACK ...|       3|2010-12-01 11:49:00|     1.65|     17920|United Kingdom|       4.95|2010|   12|      Dec|  1|      2|
|   536522|   85099B|JUMBO BAG RED RET...|       1|2010-12-01 12:49:00|     1.95| 

In [8]:
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- TotalAmount: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- MonthName: string (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Weekday: integer (nullable = true)



In [9]:
df.count()

392732

# **OVERALL ANALYSIS**

**TOTAL REVENUE**

In [10]:
df.select(round(sum("TotalAmount"),2).alias("Total Revenue")).show()

+-------------+
|Total Revenue|
+-------------+
|   8887208.89|
+-------------+



**TOTAL ORDERS**

In [11]:
from pyspark.sql.functions import count_distinct
df.select(count_distinct("InvoiceNo").alias("Total Orders")).show()

+------------+
|Total Orders|
+------------+
|       18536|
+------------+



**TOTAL CUSTOMERS**

In [12]:
df.select(count_distinct("CustomerID").alias("Total Customers")).show()

+---------------+
|Total Customers|
+---------------+
|           4339|
+---------------+



**TOTAL PRODUCTS SOLD**

In [13]:
df.select(sum("Quantity").alias("Total Quantity Sold")).show()

+-------------------+
|Total Quantity Sold|
+-------------------+
|            5165886|
+-------------------+



**AVERAGE ORDER VALUE**

In [14]:
df.groupBy("InvoiceNo").agg(sum("TotalAmount").alias("OrderValue")).select(round(avg("OrderValue"),2).alias("Average Order Value")).show()

+-------------------+
|Average Order Value|
+-------------------+
|             479.46|
+-------------------+



# **PRODUCT ANALYSIS**

**TOP 10 PRODUCTS BY REVENUE**

In [15]:
df.groupBy("Description").agg(round(sum("TotalAmount"),2).alias("Revenue")).orderBy(desc("Revenue")).show(10,False)

+----------------------------------+---------+
|Description                       |Revenue  |
+----------------------------------+---------+
|PAPER CRAFT , LITTLE BIRDIE       |168469.6 |
|REGENCY CAKESTAND 3 TIER          |142264.75|
|WHITE HANGING HEART T-LIGHT HOLDER|100392.1 |
|JUMBO BAG RED RETROSPOT           |85040.54 |
|MEDIUM CERAMIC TOP STORAGE JAR    |81416.73 |
|POSTAGE                           |77803.96 |
|PARTY BUNTING                     |68785.23 |
|ASSORTED COLOUR BIRD ORNAMENT     |56413.03 |
|Manual                            |53419.93 |
|RABBIT NIGHT LIGHT                |51251.24 |
+----------------------------------+---------+
only showing top 10 rows


**BOTTOM 10 PRODUCTS BY REVENUE**

In [16]:
df.groupBy("Description").agg(round(sum("TotalAmount"),2).alias("Revenue")).orderBy("Revenue").show(10,False)

+-----------------------------------+-------+
|Description                        |Revenue|
+-----------------------------------+-------+
|PADS TO MATCH ALL CUSHIONS         |0.0    |
|HEN HOUSE W CHICK IN NEST          |0.42   |
|SET 12 COLOURING PENCILS DOILEY    |0.65   |
|VINTAGE BLUE TINSEL REEL           |0.84   |
|PURPLE FRANGIPANI HAIRCLIP         |0.85   |
|PINK CRYSTAL GUITAR PHONE CHARM    |0.85   |
|CAT WITH SUNGLASSES BLANK CARD     |0.95   |
|HAPPY BIRTHDAY CARD TEDDY/CAKE     |0.95   |
|60 GOLD AND SILVER FAIRY CAKE CASES|1.1    |
|DUSTY PINK CHRISTMAS TREE 30CM     |1.25   |
+-----------------------------------+-------+
only showing top 10 rows


**TOP 10 PRODUCTS BY QUANTITY SOLD**

In [17]:
df.groupBy("Description").agg(sum("Quantity").alias("Quantity Sold")).orderBy(desc("Quantity Sold")).show(10,False)

+----------------------------------+-------------+
|Description                       |Quantity Sold|
+----------------------------------+-------------+
|PAPER CRAFT , LITTLE BIRDIE       |80995        |
|MEDIUM CERAMIC TOP STORAGE JAR    |77916        |
|WORLD WAR 2 GLIDERS ASSTD DESIGNS |54319        |
|JUMBO BAG RED RETROSPOT           |46078        |
|WHITE HANGING HEART T-LIGHT HOLDER|36706        |
|ASSORTED COLOUR BIRD ORNAMENT     |35263        |
|PACK OF 72 RETROSPOT CAKE CASES   |33670        |
|POPCORN HOLDER                    |30919        |
|RABBIT NIGHT LIGHT                |27153        |
|MINI PAINT SET VINTAGE            |26076        |
+----------------------------------+-------------+
only showing top 10 rows


**BOTTOM 10 PRODUCTS SOLD BY QUANTITY**

In [18]:
df.groupBy("Description").agg(sum("Quantity").alias("Quantity Sold")).orderBy("Quantity Sold").show(10,False)

+-----------------------------------+-------------+
|Description                        |Quantity Sold|
+-----------------------------------+-------------+
|PINK POLKADOT KIDS BAG             |1            |
|FRYING PAN RED POLKADOT            |1            |
|SET 10 CARDS HANGING BAUBLES 17080 |1            |
|MARIE ANTOIENETT TRINKET BOX GOLD  |1            |
|MIDNIGHT BLUE CRYSTAL DROP EARRINGS|1            |
|TEA TIME BREAKFAST BASKET          |1            |
|PURPLE FRANGIPANI HAIRCLIP         |1            |
|BLUE PADDED SOFT MOBILE            |1            |
|EASTER CRAFT IVY WREATH WITH CHICK |1            |
|CROCHET DOG KEYRING                |1            |
+-----------------------------------+-------------+
only showing top 10 rows


# **CUSTOMER ANALYSIS**

**TOP 10 AND BOTTOM 10 CUSTOMERS BY REVENUE**

In [19]:
customer_revenue = df.groupBy("CustomerID").agg(round(sum("TotalAmount"),2).alias("Revenue"))
customer_revenue.orderBy(desc("Revenue")).show(10)
customer_revenue.orderBy("Revenue").show(10)

+----------+---------+
|CustomerID|  Revenue|
+----------+---------+
|     14646|280206.02|
|     18102| 259657.3|
|     17450|194390.79|
|     16446| 168472.5|
|     14911|143711.17|
|     12415|124914.53|
|     14156|117210.08|
|     17511| 91062.38|
|     16029| 80850.84|
|     12346|  77183.6|
+----------+---------+
only showing top 10 rows
+----------+-------+
|CustomerID|Revenue|
+----------+-------+
|     13256|    0.0|
|     16738|   3.75|
|     14792|    6.2|
|     16454|    6.9|
|     17956|  12.75|
|     16878|   13.3|
|     17763|   15.0|
|     15823|   15.0|
|     13307|   15.0|
|     16093|   17.0|
+----------+-------+
only showing top 10 rows


**TOP 10 AND BOTTOM 10 CUSTOMERS BY ORDERS**

In [20]:
df.groupBy("CustomerID").agg(count_distinct("InvoiceNo").alias("Orders")).orderBy(desc("Orders")).show(10)
df.groupBy("CustomerID").agg(count_distinct("InvoiceNo").alias("Orders")).orderBy("Orders").show(10)

+----------+------+
|CustomerID|Orders|
+----------+------+
|     12748|   210|
|     14911|   201|
|     17841|   124|
|     13089|    97|
|     14606|    93|
|     15311|    91|
|     12971|    86|
|     14646|    74|
|     16029|    63|
|     13408|    62|
+----------+------+
only showing top 10 rows
+----------+------+
|CustomerID|Orders|
+----------+------+
|     14423|     1|
|     14420|     1|
|     14536|     1|
|     15790|     1|
|     14148|     1|
|     13832|     1|
|     15004|     1|
|     15447|     1|
|     17172|     1|
|     15254|     1|
+----------+------+
only showing top 10 rows


# **COUNTRY ANALYSIS**

**REVENUE BY COUNTRY**

In [21]:
df.groupBy("Country").agg(round(sum("TotalAmount"),2).alias("Revenue")).orderBy(desc("Revenue")).show(5,False)
df.groupBy("Country").agg(round(sum("TotalAmount"),2).alias("Revenue")).orderBy("Revenue").show(5,False)

+--------------+----------+
|Country       |Revenue   |
+--------------+----------+
|United Kingdom|7285024.64|
|Netherlands   |285446.34 |
|EIRE          |265262.46 |
|Germany       |228678.4  |
|France        |208934.31 |
+--------------+----------+
only showing top 5 rows
+--------------+-------+
|Country       |Revenue|
+--------------+-------+
|Saudi Arabia  |145.92 |
|Bahrain       |548.4  |
|Czech Republic|826.74 |
|RSA           |1002.31|
|Brazil        |1143.6 |
+--------------+-------+
only showing top 5 rows


**ORDERS BY COUNTRY**

In [22]:
df.groupBy("Country").agg(count_distinct("InvoiceNo").alias("Orders")).orderBy(desc("Orders")).show(5,False)
df.groupBy("Country").agg(count_distinct("InvoiceNo").alias("Orders")).orderBy("Orders").show(5,False)

+--------------+------+
|Country       |Orders|
+--------------+------+
|United Kingdom|16649 |
|Germany       |457   |
|France        |389   |
|EIRE          |260   |
|Belgium       |98    |
+--------------+------+
only showing top 5 rows
+--------------+------+
|Country       |Orders|
+--------------+------+
|RSA           |1     |
|Saudi Arabia  |1     |
|Brazil        |1     |
|Lebanon       |1     |
|Czech Republic|2     |
+--------------+------+
only showing top 5 rows


**CUSTOMERS BY COUNTRY**

In [23]:
df.groupBy("Country").agg(count_distinct("CustomerID").alias("Customers")).orderBy(desc("Customers")).show(5,False)
df.groupBy("Country").agg(count_distinct("CustomerID").alias("Customers")).orderBy("Customers").show(5,False)

+--------------+---------+
|Country       |Customers|
+--------------+---------+
|United Kingdom|3921     |
|Germany       |94       |
|France        |87       |
|Spain         |30       |
|Belgium       |25       |
+--------------+---------+
only showing top 5 rows
+------------------+---------+
|Country           |Customers|
+------------------+---------+
|European Community|1        |
|Singapore         |1        |
|RSA               |1        |
|Saudi Arabia      |1        |
|Lithuania         |1        |
+------------------+---------+
only showing top 5 rows


# **TIME ANALYSIS**

In [26]:
df.show()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+----+-----+---------+---+-------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|TotalAmount|Year|Month|MonthName|Day|Weekday|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+----+-----+---------+---+-------+
|   536373|    21071|VINTAGE BILLBOARD...|       6|2010-12-01 09:02:00|     1.06|     17850|United Kingdom|       6.36|2010|   12|      Dec|  1|      2|
|   536373|   84029G|KNITTED UNION FLA...|       6|2010-12-01 09:02:00|     3.39|     17850|United Kingdom|      20.34|2010|   12|      Dec|  1|      2|
|   536412|    20727|LUNCH BAG  BLACK ...|       3|2010-12-01 11:49:00|     1.65|     17920|United Kingdom|       4.95|2010|   12|      Dec|  1|      2|
|   536522|   85099B|JUMBO BAG RED RET...|       1|2010-12-01 12:49:00|     1.95| 

**YEARLY REVENUE**

In [40]:
df.groupBy("Year").agg(round(sum("TotalAmount"),2).alias("Revenue")).show()

+----+----------+
|Year|   Revenue|
+----+----------+
|2010| 570422.73|
|2011|8316786.16|
+----+----------+



**MONTHLY REVENUE**

In [41]:
df.groupBy("Year","MonthName").agg(round(sum("TotalAmount"),2).alias("Revenue")).orderBy(desc("Revenue")).show()

+----+---------+----------+
|Year|MonthName|   Revenue|
+----+---------+----------+
|2011|      Nov|1156205.61|
|2011|      Oct|1035642.45|
|2011|      Sep|  950690.2|
|2011|      May| 677355.15|
|2011|      Jun| 660046.05|
|2011|      Aug| 644051.04|
|2011|      Jul|  598962.9|
|2011|      Mar| 594081.76|
|2010|      Dec| 570422.73|
|2011|      Jan| 568101.31|
|2011|      Dec| 517190.44|
|2011|      Apr| 468374.33|
|2011|      Feb| 446084.92|
+----+---------+----------+



# **RANKINGS**

**RANKING CUSTOMERS BY REVENUE**

In [61]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

window_spec = Window.orderBy(desc("Revenue"))
customer_revenue.withColumn("Rank",dense_rank().over(window_spec)).show()

+----------+---------+----+
|CustomerID|  Revenue|Rank|
+----------+---------+----+
|     14646|280206.02|   1|
|     18102| 259657.3|   2|
|     17450|194390.79|   3|
|     16446| 168472.5|   4|
|     14911|143711.17|   5|
|     12415|124914.53|   6|
|     14156|117210.08|   7|
|     17511| 91062.38|   8|
|     16029| 80850.84|   9|
|     12346|  77183.6|  10|
|     16684| 66653.56|  11|
|     14096| 65164.79|  12|
|     13694| 65039.62|  13|
|     15311| 60632.75|  14|
|     13089| 58762.08|  15|
|     17949| 58510.48|  16|
|     15769| 56252.72|  17|
|     15061| 54534.14|  18|
|     14298|  51527.3|  19|
|     14088| 50491.81|  20|
+----------+---------+----+
only showing top 20 rows


**RANK PRODUCTS BY REVENUE**

In [65]:
from pyspark.sql.functions import rank

product_revenue = df.groupBy("Description").agg(round(sum("TotalAmount"), 2).alias("Revenue"))

window_spec = Window.orderBy(desc("Revenue"))
product_revenue.withColumn("Rank",rank().over(window_spec)).show(truncate=False)

+----------------------------------+---------+----+
|Description                       |Revenue  |Rank|
+----------------------------------+---------+----+
|PAPER CRAFT , LITTLE BIRDIE       |168469.6 |1   |
|REGENCY CAKESTAND 3 TIER          |142264.75|2   |
|WHITE HANGING HEART T-LIGHT HOLDER|100392.1 |3   |
|JUMBO BAG RED RETROSPOT           |85040.54 |4   |
|MEDIUM CERAMIC TOP STORAGE JAR    |81416.73 |5   |
|POSTAGE                           |77803.96 |6   |
|PARTY BUNTING                     |68785.23 |7   |
|ASSORTED COLOUR BIRD ORNAMENT     |56413.03 |8   |
|Manual                            |53419.93 |9   |
|RABBIT NIGHT LIGHT                |51251.24 |10  |
|CHILLI LIGHTS                     |46265.11 |11  |
|PAPER CHAIN KIT 50'S CHRISTMAS    |42584.13 |12  |
|PICNIC BASKET WICKER 60 PIECES    |39619.5  |13  |
|BLACK RECORD COVER FRAME          |39045.8  |14  |
|JUMBO BAG PINK POLKADOT           |37254.36 |15  |
|DOORMAT KEEP CALM AND COME IN     |35880.85 |16  |
|SPOTTY BUNT

**TOP PRODUCTS BY EACH COUNTRY**

In [67]:
country_product = df.groupBy("Country", "Description").agg(round(sum("TotalAmount"), 2).alias("Revenue"))

window_spec = Window.partitionBy("Country").orderBy(desc("Revenue"))
country_product.withColumn("Rank",row_number().over(window_spec)).filter(col("Rank") == 1).show(truncate=False)

+------------------+----------------------------------+-------+----+
|Country           |Description                       |Revenue|Rank|
+------------------+----------------------------------+-------+----+
|Australia         |RABBIT NIGHT LIGHT                |3375.84|1   |
|Austria           |POSTAGE                           |1456.0 |1   |
|Bahrain           |ICE CREAM SUNDAE LIP GLOSS        |120.0  |1   |
|Belgium           |POSTAGE                           |4269.0 |1   |
|Brazil            |REGENCY CAKESTAND 3 TIER          |175.2  |1   |
|Canada            |POSTAGE                           |550.94 |1   |
|Channel Islands   |REGENCY CAKESTAND 3 TIER          |517.8  |1   |
|Cyprus            |RUSTIC  SEVENTEEN DRAWER SIDEBOARD|580.0  |1   |
|Czech Republic    |ROUND SNACK BOXES SET OF4 WOODLAND|70.8   |1   |
|Denmark           |POSTAGE                           |744.0  |1   |
|EIRE              |REGENCY CAKESTAND 3 TIER          |7337.55|1   |
|European Community|POSTAGE       